# 21 · Inferencia estadística y confiabilidad probabilística de la segmentación porcina

**CPU · Notebook independiente · FaroPigSeg · Resultados reales, sin entrenar ni repetir inferencias.**

Este notebook integra tres resultados: (i) una **figura principal de confiabilidad en cuatro paneles** reconstruida con las 25.763 detecciones de prueba del Notebook 14 (37.936 incluyendo validación), siguiendo la referencia editorial seleccionada; (ii) una segunda figura de calibración **operativa** (filtrando por confianza original ≥ 0,25); y (iii) contrastes **apareados por fecha** de mAP de máscaras, error de conteo e IoU de máscaras emparejadas.

**Regla estadística:** las 373 imágenes no se tratan como observaciones independientes para las pruebas. Se agregan a las **cinco fechas** del conjunto de prueba. La prueba de permutación exacta con cinco fechas tiene resolución limitada y es **exploratoria**, porque fechas y sesiones verdaderas no son necesariamente independientes. Las métricas de IoU se condicionan a que exista emparejamiento de máscaras. El AP por fecha NO sustituye el AP global.

La figura de calibración muestra **detecciones propuestas**, no fiabilidad píxel a píxel; los falsos negativos no forman parte del ECE. El test ya se ha utilizado en estudios previos, por lo que no es una nueva validación externa.

## A. Dependencias y fuentes congeladas

En Colab, seleccionar **Entorno de ejecución → Cambiar tipo de entorno → CPU / Sin acelerador**. Ejecutar todas las celdas. El notebook escribe únicamente sus propios archivos `CERDOS_21_*` en las carpetas de resultados.

In [ ]:
%pip -q install "numpy>=1.24,<3" "pandas>=2,<3" "matplotlib>=3.7" "openpyxl>=3.1"

In [ ]:
from google.colab import drive, files
from pathlib import Path
from datetime import datetime, timezone
from itertools import product
import hashlib, json, zipfile, platform
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Patch
from matplotlib.lines import Line2D

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/Proyecto de investigación: Visión Artificial y Deep Learning en Cerdos 🐖')
assert ROOT.is_dir(), f'No se encontró la carpeta del proyecto: {ROOT}'
MET = ROOT/'03_Resultados'/'03_Metricas'
FIG = ROOT/'03_Resultados'/'04_Figuras'
REP = ROOT/'03_Resultados'/'05_Reportes'
EXP = ROOT/'05_Exportaciones_ZIP'
for p in [MET, FIG, REP, EXP]: p.mkdir(parents=True,exist_ok=True)

F09 = MET/'CERDOS_09_Comparacion_Armonizada_COCO.xlsx'
F11 = MET/'CERDOS_11_Confiabilidad_Geometrica.xlsx'
F14 = MET/'CERDOS_14_Calibracion_Probabilistica.xlsx'
F15 = MET/'CERDOS_15_Correccion_Conteo_Val.xlsx'
GT = REP/'CERDOS_09_COCO_Armonizado'/'CERDOS_09_GT_test_coco.json'
SOURCES = [F09,F11,F14,F15,GT]
for path in SOURCES:
    if not path.is_file(): raise FileNotFoundError(f'Falta fuente validada: {path}')

def sha256(path):
    h=hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda:f.read(4*1024*1024),b''): h.update(block)
    return h.hexdigest()
HASHES = {str(p.relative_to(ROOT)):sha256(p) for p in SOURCES}
Y, M = 'YOLO11n-seg','Mask R-CNN'
def normalize_model(x):
    s=str(x)
    if s.startswith('YOLO'):return Y
    if s.startswith('Mask R-CNN'):return M
    raise ValueError('Modelo desconocido: '+s)
print('✅ Archivos originales localizados y huellas SHA-256 registradas. CPU, sin entrenamiento.')

## B. Curvas de calibración desde detecciones reales

La figura principal conserva la población original del Notebook 14 (**scores ≥ 0,001**), con 10 intervalos equiespaciados. La segunda es un análisis de sensibilidad con **score original ≥ 0,25**, tomando la misma población antes y después de Platt. Todas las etiquetas de acierto se fijaron por emparejamiento uno a uno a IoU de máscara ≥ 0,50. No se modifican las probabilidades, los umbrales ni las detecciones durante esta fase.

In [ ]:
D = pd.read_excel(F14, sheet_name='Detecciones_Prob')
required={'modelo','particion','imagen_id','fecha','score_raw','score_calibrado','correcta_IoU05'}
assert required.issubset(D.columns),f'Faltan columnas: {required-set(D.columns)}'
D['modelo']=D['modelo'].map(normalize_model)
D=D[D['particion'].eq('prueba')].copy()
assert len(D)==25763 and set(D['modelo'])=={Y,M}
assert D['imagen_id'].nunique()==373 and D['fecha'].nunique()==5
assert D['correcta_IoU05'].isin([0,1]).all()
assert D[['score_raw','score_calibrado']].notna().all().all()
assert D[['score_raw','score_calibrado']].ge(0).all().all()
assert D[['score_raw','score_calibrado']].le(1).all().all()

BIN_COUNT=10
POPULATIONS=[('Global · all detections (raw ≥ 0.001)',0.001),
             ('Operational · fixed raw ≥ 0.25',0.25)]
def confidence_bins(d, field, n=10):
    y=d['correcta_IoU05'].to_numpy(dtype=float)
    probs=d[field].to_numpy(dtype=float)
    assert len(probs)>0
    bins=np.minimum((np.maximum(probs,0)*n).astype(int),n-1)
    rows=[];ece=0.
    for i in range(n):
        selected=bins==i
        count=int(selected.sum())
        mean_conf=float(probs[selected].mean()) if count else np.nan
        empirical=float(y[selected].mean()) if count else np.nan
        if count: ece+=count/len(probs)*abs(empirical-mean_conf)
        rows.append({'bin':i,'midpoint':(i+.5)/n,'n':count,
                     'mean_confidence':mean_conf,'observed_precision':empirical,
                     'bin_lower':i/n,'bin_upper':(i+1)/n})
    brier=float(np.mean((probs-y)**2))
    cp=np.clip(probs,1e-9,1-1e-9)
    logloss=float(-np.mean(y*np.log(cp)+(1-y)*np.log1p(-cp)))
    return pd.DataFrame(rows),float(ece),brier,logloss

calibration_rows=[]; curve_rows=[]
for population,cut in POPULATIONS:
    for model in (Y,M):
        base=D[D.modelo.eq(model)&D.score_raw.ge(cut)].copy()
        for status,field in [('Uncalibrated','score_raw'),('Platt calibrated','score_calibrado')]:
            curves,ece,brier,ll=confidence_bins(base,field,BIN_COUNT)
            curves['population']=population;curves['model']=model;curves['state']=status
            curve_rows.extend(curves.to_dict('records'))
            calibration_rows.append({'population':population,'model':model,'state':status,
                 'n_detections':len(base),'n_correct':int(base.correcta_IoU05.sum()),
                 'ECE_10':ece,'Brier_score':brier,'Log_loss':ll})
CAL=pd.DataFrame(calibration_rows);BINS=pd.DataFrame(curve_rows)
# Verificar exactamente las cifras históricas originales de Notebook 14 (sin recalibrar):
ref=CAL[CAL.population.eq(POPULATIONS[0][0])]
expected={(Y,'Uncalibrated'):0.025999212375142612,(Y,'Platt calibrated'):0.011201570044382624,
          (M,'Uncalibrated'):0.0705,(M,'Platt calibrated'):0.0118}
for (model,status),val in expected.items():
    found=float(ref[(ref.model.eq(model))&(ref.state.eq(status))].ECE_10.iloc[0])
    assert abs(found-val)<0.001, f'Inconsistencia de calibración {model}/{status}: {found}'
print('✅ ECE recalculado con las detecciones reales del Notebook 14:')
display(CAL[['population','model','state','n_detections','ECE_10','Brier_score','Log_loss']])

## C. Figura de revista: cuatro paneles, antes y después de la calibración

**Azul:** precisión observada en cada intervalo. **Coral:** discrepancia entre precisión observada y confianza media. **Diagonal discontinua:** calibración perfecta. La población y los ECE se recalculan, no se insertan manualmente. Cada figura se exporta a **PNG 600 dpi, PDF y SVG**. El ECE depende de la población de detecciones y del binning, no representa falsos negativos ni incertidumbre a nivel de píxel.

In [ ]:
from matplotlib.ticker import MultipleLocator, FormatStrFormatter
BLUE='#648DB1'; GAP='#E28D8C'; INK='#1C2833'; GRID='#E7EBEF'
mpl.rcParams.update({'font.family':'DejaVu Serif','font.size':10.5,
                    'axes.labelsize':11,'axes.titlesize':12,
                    'pdf.fonttype':42,'ps.fonttype':42,'svg.fonttype':'none',
                    'figure.facecolor':'white','savefig.facecolor':'white'})

def plot_reliability(population,label,suffix):
    fig,axes=plt.subplots(2,2,figsize=(12.9,9.4),sharex=True,sharey=True)
    letters=['a','b','c','d']; k=0
    for i,model in enumerate((Y,M)):
        for j,state in enumerate(('Uncalibrated','Platt calibrated')):
            ax=axes[i,j]
            selected=BINS[(BINS.population.eq(population))&(BINS.model.eq(model))&(BINS.state.eq(state))]
            summ=CAL[(CAL.population.eq(population))&(CAL.model.eq(model))&(CAL.state.eq(state))].iloc[0]
            assert len(selected)==BIN_COUNT
            ax.plot([0,1],[0,1],ls='--',color=INK,lw=1.1,zorder=4)
            for _,r in selected.iterrows():
                x=float(r.midpoint); n=int(r.n)
                if n==0: continue
                obs=float(r.observed_precision);conf=float(r.mean_confidence)
                w=.085
                ax.bar(x,obs,width=w,color=BLUE,edgecolor='white',linewidth=.5,alpha=.96,zorder=2)
                if abs(obs-conf)>1e-7:
                    bottom=min(obs,conf)
                    height=abs(obs-conf)
                    ax.add_patch(Rectangle((x-w/2,bottom),w,height,facecolor=GAP,
                                     alpha=.94,edgecolor='white',linewidth=.45,zorder=3))
            ax.set_xlim(0,1);ax.set_ylim(0,1.02)
            ax.xaxis.set_major_locator(MultipleLocator(.2))
            ax.yaxis.set_major_locator(MultipleLocator(.2))
            ax.xaxis.set_major_formatter(FormatStrFormatter('%.1f'))
            ax.yaxis.set_major_formatter(FormatStrFormatter('%.1f'))
            ax.grid(color=GRID,linewidth=.7,zorder=0)
            ax.set_axisbelow(True)
            ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
            ax.set_title(f'({letters[k]}) {model} — {state}  |  ECE = {summ.ECE_10:.4f}',
                         weight='bold',fontsize=11.4,pad=11)
            ax.text(.98,.045,f'n = {int(summ.n_detections):,}',transform=ax.transAxes,
                    ha='right',va='bottom',fontsize=9,color='#505E69')
            if j==0:ax.set_ylabel('Observed detection precision',labelpad=8)
            if i==1:ax.set_xlabel('Predicted confidence',labelpad=6)
            k+=1
    fig.suptitle('Reliability of pig instance detection confidences',
                 fontsize=16.8,weight='bold',y=.99)
    fig.text(.5,.946,f'Mask IoU ≥ 0.50 · {label} · 10 equal-width confidence bins',
             ha='center',fontsize=10,color='#526171')
    legend=[Line2D([0],[0],color=INK,lw=1.2,ls='--',label='Perfect calibration'),
            Patch(facecolor=BLUE,label='Observed precision'),
            Patch(facecolor=GAP,label='Calibration gap')]
    fig.legend(handles=legend,loc='lower center',bbox_to_anchor=(.5,.045),
               ncol=3,frameon=False,fontsize=10.5)
    fig.text(.5,.015,'Detection-level reliability; false negatives excluded. Test previously inspected.',
             ha='center',fontsize=8.8,color='#677481')
    fig.subplots_adjust(top=.89,bottom=.15,left=.075,right=.984,hspace=.32,wspace=.18)
    base=FIG/f'CERDOS_21_Reliability_{suffix}'
    paths=[]
    for ext in ('png','pdf','svg'):
        path=Path(str(base)+'.'+ext)
        fig.savefig(path,dpi=600,facecolor='white')
        paths.append(path)
    plt.show();plt.close(fig)
    return paths

MAIN_PLOTS=plot_reliability(POPULATIONS[0][0],'all retained detections (raw confidence ≥ 0.001)','Global')
OP_PLOTS=plot_reliability(POPULATIONS[1][0],'operational subset (fixed raw confidence ≥ 0.25)','Operational')
print('✅ Se generaron dos figuras profesionales de cuatro paneles; no se copiaron imágenes de ejemplo.')

## D. Inferencia estadística apareada por fecha

**Tres contrastes exploratorios (cinco fechas):** (1) diferencia de AP@50:95 de máscaras, (2) diferencia en MAE de conteo usando umbrales seleccionados **solo en validación** y (3) diferencia de IoU medio de máscaras emparejadas por imagen. Se usa una prueba exacta de **32 inversiones de signo** y bootstrap apareado de cinco fechas. Se reportan estadísticos y corrección de Holm para tres pruebas, pero **no se interpreta la fecha como una sesión independiente demostrada**.

El cambio de signo se define para que un valor positivo favorezca a YOLO: `AP_YOLO – AP_Mask`, `MAE_Mask – MAE_YOLO`, `IoU_YOLO – IoU_Mask`. Las medias por fecha son no ponderadas; no equivalen a mAP global ni a media ponderada por instancia.

In [ ]:
AP=pd.read_excel(F09,sheet_name='Por_Fecha_COCO')
PAIR=pd.read_excel(F11,sheet_name='Pares_Geometricos')
COUNT=pd.read_excel(F15,sheet_name='Por_imagen')
for q in [AP,PAIR,COUNT]:q['modelo']=q['modelo'].map(normalize_model)
assert len(AP)==10 and AP.groupby('modelo')['fecha'].nunique().eq(5).all()
APD=AP.pivot(index='fecha',columns='modelo',values='mAP50_95_mascaras')
APD=APD[[Y,M]].dropna().sort_index()
AP_DELTA=APD[Y]-APD[M]

COUNT=COUNT[(COUNT['split']=='test') & (COUNT['escenario']=='Elegido en val')].copy()
assert COUNT.groupby('modelo')['imagen_id'].nunique().eq(373).all()
CPIV=COUNT.pivot(index='imagen_id',columns='modelo',values='error_abs')
CFECHA=COUNT.drop_duplicates('imagen_id').set_index('imagen_id')['fecha']
assert len(CPIV)==373 and CPIV.notna().all().all()
CPIV['fecha']=CPIV.index.map(CFECHA)
# Cambio positivo = error mayor en Mask R-CNN que en YOLO.
COUNT_DELTA=CPIV.assign(delta=CPIV[M]-CPIV[Y]).groupby('fecha')['delta'].mean().sort_index()

PAIR['image_id']=PAIR['image_id'].astype(int)
PIM=PAIR.groupby(['image_id','modelo'])['mask_iou'].mean().unstack('modelo')
PIM=PIM[[Y,M]].dropna() # condiciona a que ambos tengan predicciones emparejadas
DATE_BY_IMG=COUNT.drop_duplicates('imagen_id').set_index('imagen_id')['fecha']
PIM['fecha']=PIM.index.map(DATE_BY_IMG)
assert PIM['fecha'].notna().all() and PIM['fecha'].nunique()==5
IOU_DELTA=PIM.assign(delta=PIM[Y]-PIM[M]).groupby('fecha')['delta'].mean().sort_index()

DATE_INDEX=sorted(AP_DELTA.index)
assert DATE_INDEX==sorted(COUNT_DELTA.index)==sorted(IOU_DELTA.index)
TESTDATES=pd.DataFrame({'date':DATE_INDEX,'delta_mask_AP':AP_DELTA.reindex(DATE_INDEX).values,
    'delta_counting_MAE':COUNT_DELTA.reindex(DATE_INDEX).values,
    'delta_conditional_mask_IoU':IOU_DELTA.reindex(DATE_INDEX).values})
assert len(TESTDATES)==5 and TESTDATES.iloc[:,1:].notna().all().all()
print('✅ Diferencias apareadas calculadas en cinco fechas. IoU solo sobre máscaras emparejadas.')
display(TESTDATES)

In [ ]:
SEED=2026
rng=np.random.default_rng(SEED)
ITERATIONS=20000

def exact_signflip_paired(deltas):
    x=np.asarray(deltas,dtype=float)
    assert len(x)==5 and np.isfinite(x).all()
    observed=float(x.mean())
    permutation_means=np.asarray([np.mean(x*np.asarray(s)) for s in product((-1,1),repeat=len(x))])
    p_two=float(np.mean(np.abs(permutation_means)>=abs(observed)-1e-12))
    return observed,p_two

def bootstrap_date_means(deltas,n=ITERATIONS):
    x=np.asarray(deltas,dtype=float)
    idx=rng.integers(0,len(x),size=(n,len(x)))
    means=x[idx].mean(axis=1)
    return float(np.quantile(means,.025)),float(np.quantile(means,.975))

METRICS=[('delta_mask_AP','Mask AP@50:95','percentage points',100.),
         ('delta_counting_MAE','Counting MAE','pigs per image',1.),
         ('delta_conditional_mask_IoU','Matched-mask IoU','percentage points',100.)]
rows=[]
for col,label,unit,mult in METRICS:
    x=TESTDATES[col].to_numpy(dtype=float)
    effect,p=exact_signflip_paired(x)
    lo,hi=bootstrap_date_means(x)
    rows.append({'metric':label,'unit':unit,'n_dates':5,'mean_paired_difference':effect*mult,
                 'CI_boot_2.5':lo*mult,'CI_boot_97.5':hi*mult,
                 'p_exact_two_sided':p,'signs_positive':int(np.sum(x>0)),
                 'signs_negative':int(np.sum(x<0)),
                 'analysis':'equal-weighted date-level deltas; exploratory'})
INF=pd.DataFrame(rows)
# Multiplicidad: Holm step-down para las tres pruebas exploratorias.
p=INF['p_exact_two_sided'].to_numpy(dtype=float);order=np.argsort(p)
adjusted=np.empty_like(p);prev=0.
for rank,i in enumerate(order):
    prev=max(prev,(len(p)-rank)*p[i]);adjusted[i]=min(1.,prev)
INF['p_Holm']=adjusted
print('✅ Estadísticos exploratorios, p exactos de dos colas y corrección de Holm:')
display(INF)
assert (INF['p_exact_two_sided']>=.0625-1e-12).all(), 'Con cinco fechas, el p bilateral mínimo esperado es 0.0625.'

## E. Figura inferencial complementaria y exportaciones

La figura presenta diferencias medias apareadas **con el sentido favorable a YOLO hacia la derecha**, pero en escalas separadas. Las barras expresan remuestreo de fechas (2,5–97,5 %) **exploratorio**, no intervalos de confianza confirmatorios por sesión. El resultado principal sigue siendo la evaluación armonizada del Notebook 09, no estos contrastes post hoc.

In [ ]:
fig,axes=plt.subplots(3,1,figsize=(10.8,8.0))
for i,(col,label,unit,mult) in enumerate(METRICS):
    ax=axes[i];r=INF.iloc[i]
    x=float(r.mean_paired_difference);lo=float(r['CI_boot_2.5']);hi=float(r['CI_boot_97.5'])
    ax.axvline(0,color=INK,ls='--',lw=1)
    ax.errorbar([x],[0],xerr=np.array([[x-lo],[hi-x]]),fmt='o',color=BLUE,
                capsize=5,markersize=8,elinewidth=2)
    ax.scatter(TESTDATES[col].to_numpy()*mult,np.repeat(-.15,5),s=24,
               c=GAP,alpha=.85,label='Individual dates' if i==0 else None)
    ax.set_ylim(-.36,.35);ax.set_yticks([])
    ax.set_title(f'{label} · exact two-sided p={r.p_exact_two_sided:.4f}, Holm={r.p_Holm:.4f}',
                 loc='left',fontsize=12,pad=10)
    ax.set_xlabel(f'YOLO advantage ({unit}); positive → YOLO')
    ax.grid(axis='x',alpha=.2)
    ax.spines[['top','right','left']].set_visible(False)
fig.suptitle('Paired differences across five capture dates',fontsize=16,weight='bold',y=.99)
fig.text(.5,.007,'Exploratory date bootstrap (20,000 draws). Dates may not represent independent sessions.',
         ha='center',fontsize=9,color='#526171')
fig.subplots_adjust(hspace=.76,top=.89,bottom=.13,left=.085,right=.98)
INFERENCE_BASE=FIG/'CERDOS_21_Inferencia_Pareada_Fecha'
INFER_PLOTS=[]
for ext in ('png','pdf','svg'):
    p=Path(str(INFERENCE_BASE)+'.'+ext);fig.savefig(p,dpi=600,facecolor='white');INFER_PLOTS.append(p)
plt.show();plt.close(fig)

EXCEL=MET/'CERDOS_21_Inferencia_Calibracion.xlsx'
REPORT=REP/'CERDOS_21_Inferencia_Protocolo.json'
LIMITATIONS=[
 'Resultados test post hoc: no nueva evaluación externa; test inspeccionado en notebooks previos.',
 'Solo cinco fechas; agrupaciones verdaderamente independientes por sesión/granja no verificadas.',
 'Prueba exacta de 32 permutaciones requiere intercambio de signos bajo hipótesis nula; lectura exploratoria.',
 'Bootstrap de fecha (n=5) solo descriptivo de sensibilidad; sus percentiles NO constituyen prueba confirmatoria.',
 'AP por fecha de igual peso no representa mAP global por COCOeval; no mezclar ambos estimandos.',
 'IoU se calcula solo sobre máscaras emparejadas en ambos modelos, excluyendo falsos negativos.',
 'ECE sobre detecciones propuestas de IoU >= .50; no incluye falsos negativos ni incertidumbre píxel a píxel.',
 'La calibración con Platt y el umbral de conteo fueron ajustados en validación, no en prueba.',
 'La población operativa se define por score RAW >=0.25 tanto antes como después de calibración.',
 'No confundir cinco fechas con cinco granjas o sesiones garantizadas independientes.',
 'No usar estos valores p post hoc como prueba de superioridad general entre arquitecturas.'
]
with pd.ExcelWriter(EXCEL,engine='openpyxl') as writer:
    CAL.to_excel(writer,sheet_name='Calibracion_ECE',index=False)
    BINS[BINS.population.eq(POPULATIONS[0][0])].to_excel(writer,sheet_name='Bines_Global',index=False)
    BINS[BINS.population.eq(POPULATIONS[1][0])].to_excel(writer,sheet_name='Bines_Operativo',index=False)
    TESTDATES.to_excel(writer,sheet_name='Diferencias_por_fecha',index=False)
    INF.to_excel(writer,sheet_name='Inferencia_Exploratoria',index=False)
    pd.DataFrame([{'archivo':k,'SHA256':v} for k,v in HASHES.items()]).to_excel(writer,sheet_name='Trazabilidad',index=False)
    pd.DataFrame([{'limitacion':x} for x in LIMITATIONS]).to_excel(writer,sheet_name='Limitaciones',index=False)
META={'etapa':'CERDOS_21_inferencia_y_calibracion','utc':datetime.now(timezone.utc).isoformat(),
      'fuentes_sha256':HASHES,'n_fechas':5,'n_permutaciones_exactas':32,
      'iteraciones_bootstrap':ITERATIONS,'bootstrap_semilla':SEED,
      'ECE_bins':BIN_COUNT,'correcta_deteccion':'IoU_mascara>=0.50 y matching 1a1; labels de notebook14',
      'inferencia':INF.to_dict('records'),'calibracion':CAL.to_dict('records'),
      'limitaciones':LIMITATIONS,'python':platform.python_version()}
if REPORT.is_file():
    old=json.loads(REPORT.read_text(encoding='utf8'))
    if old.get('fuentes_sha256')!=HASHES:
        raise RuntimeError('Existe Notebook 21 con OTRAS fuentes. No sobrescribir: audita los datos primero.')
REPORT.write_text(json.dumps(META,ensure_ascii=False,indent=2),encoding='utf8')
ZIP=EXP/'CERDOS_21_Inferencia_y_Confiabilidad_Resultados.zip'
ARTIFACTS=[EXCEL,REPORT]+MAIN_PLOTS+OP_PLOTS+INFER_PLOTS
with zipfile.ZipFile(ZIP,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for path in ARTIFACTS:z.write(path,arcname=path.name)
with zipfile.ZipFile(ZIP) as z:assert z.testzip() is None
print('✅ Archivos exportados a Drive:')
for path in ARTIFACTS:print('  •',path.name)
print('✅ ZIP:',ZIP,'·',round(ZIP.stat().st_size/1024/1024,2),'MiB')
files.download(str(ZIP))

## F. Uso en el manuscrito

**Figura principal recomendada:** `CERDOS_21_Reliability_Global.pdf`. Figura adicional de sensibilidad: `CERDOS_21_Reliability_Operational.pdf`. La figura inferencial de diferencias por fecha puede usarse como complemento, con la advertencia de dependencia y pocos grupos.

**Pie propuesto (inglés):** *Reliability diagrams of confidence scores for proposed pig instances before and after Platt scaling. A detection was marked correct after one-to-one matching to a ground-truth instance at mask IoU ≥ 0.50. Calibration used independent validation dates; scores were evaluated on 373 previously inspected test images. Blue bars denote empirical precision and coral segments the calibration gap. ECE is calculated in ten equal-width confidence bins. These plots do not quantify pixel-wise uncertainty or account for missed ground-truth instances.*

**Precaución:** los contrastes estadísticos son exploratorios; no afirmar *statistical superiority* sin un diseño con más sesiones verificablemente independientes o un conjunto externo verdaderamente nuevo.